# Fine-Tuning Ling Tiny on Mac mini with AReno

Open this notebook in JupyterLab on an Apple Silicon Mac. Review the settings in Section 2,
then run the cells in order. **Run → Run All Cells** also starts model downloads, both
validation passes, and the full training epoch.

Follow the [Mac mini walkthrough](MAC_MINI.md) for environment setup and troubleshooting.
This recipe uses AReno's MLX backend and unquantized attention LoRA. Ling support is
experimental; end-to-end training and memory requirements on Mac mini remain unverified.

<p style="display: flex; align-items: center; gap: 10px; flex-wrap: wrap;">
  <a href="https://github.com/inclusionAI/AReno"><img src="../../docs/_static/areno-logo.svg" alt="AReno" width="115" style="vertical-align: middle;"></a>
  <a href="../../CONTRIBUTING.md" title="Contribute to the AReno community"><img src="assets/join-community.svg" alt="Join community" height="36" style="vertical-align: middle;"></a>
  <a href="https://asystem-ai.io/docs/areno/"><img src="assets/documentation.svg" alt="Documentation" height="36" style="vertical-align: middle;"></a>
  <span>⭐ <a href="https://github.com/inclusionAI/AReno"><i>Star us on GitHub</i></a></span>
</p>

To install **[AReno](https://github.com/inclusionAI/AReno)** on your local Mac, follow our [setup guide](README.md). This notebook is licensed under [Apache-2.0](../../LICENSE).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [save it](#AdapterSave) & how to [verify training result](#Inference).


## 1. Installation

Install AReno and the required dependencies into your current JupyterLab kernel environment.
Use a dedicated environment and a fresh kernel; MLX-LM is pinned to a revision containing Ling support.
The Mac path uses Metal through MLX and does not require CUDA or FlashAttention.
If you have already imported MLX or AReno before installing, restart the kernel and rerun from here.


In [ ]:
import platform
import subprocess
import sys
from pathlib import Path

if platform.system() != "Darwin" or platform.machine().lower() != "arm64":
    raise RuntimeError("Use this notebook on an Apple Silicon Mac. For CUDA, open Ling-Tiny-SFT-Colab.ipynb.")
RECIPE_SUBDIR = Path("recipes/ling-tiny-fine-tuning")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / RECIPE_SUBDIR / "recipe.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook in JupyterLab from within your local AReno checkout.")
RECIPE = ROOT / RECIPE_SUBDIR
# Install the local AReno checkout and its declared dependencies with the kernel's Python.
# psutil is used by this notebook to display unified memory capacity.
# This MLX-LM revision includes the Ling architecture missing from release 0.31.3.
MLX_LM_REVISION = "1840cbba9cb3f8125b8cf7e137606256a2f162d1"
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-e", str(ROOT), "psutil",
    f"mlx-lm @ git+https://github.com/ml-explore/mlx-lm.git@{MLX_LM_REVISION}",
], check=True)
sys.path.insert(0, str(RECIPE))
print("AReno source:", ROOT)
print("Commit:", subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip())


## 2. Configure the experiment

All stages use the same prompt template, token budgets, and validation split. Model assets
come from ModelScope by default; `MODEL_PATH` can point to an existing unquantized checkpoint.
Edit the settings in the next cell, including the dataset mirror and local output directory.
A failed download does not switch providers.

Defaults: one full epoch over 1,400 examples, batch size 4, microbatch size 1, rank 8,
constant learning rate `1e-4`, and greedy evaluation of all 100 validation examples.
These are starting settings, not an established optimal configuration.


In [ ]:
import json
from datetime import datetime, timezone

DATA_HUB = "modelscope"  # "modelscope" or "huggingface"
DATASET_ID = "AsystemAReno/Ling-Tiny-SFT"
DATA_REVISIONS = {
    "modelscope": "bef344521b1751374d3ff9da3b9f4f61706d5c8d",
    "huggingface": "83d7be9365b1de384d41c970c816698ebda29571",
}
MODEL_ID = "inclusionAI/Ling-3.0-tiny"
MODEL_REVISION = "0a9ee734a409577fa0cff0ff48b48df008e7d8d4"
MODEL_PATH = ""  # Optional local checkpoint directory.
EPOCHS = 1
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
LORA_RANK = 8
LORA_ALPHA = 16
SEED = 42
MAX_PROMPT_TOKENS = 4096
MAX_NEW_TOKENS = 1024
MAX_CONTEXT_LEN = MAX_PROMPT_TOKENS + MAX_NEW_TOKENS

run_name = "ling-tiny-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = (ROOT / "runs" / run_name).resolve()  # Edit to choose another local output directory.
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run outputs:", RUN_DIR)


In [ ]:
import psutil

BACKEND = "mlx"
# Probe in a child process so the notebook does not retain a Metal runtime.
probe = """
import importlib.util
import json
from importlib.metadata import version
import mlx.core as mx
if not mx.metal.is_available():
    raise RuntimeError("MLX Metal is unavailable. Use a native arm64 Python on a supported macOS version.")
if importlib.util.find_spec("mlx_lm.models.bailing_moe_v3") is None:
    raise RuntimeError("Ling support is missing. Rerun installation with the pinned MLX-LM revision.")
print(json.dumps({"metal_available": True, "mlx": version("mlx"), "mlx_lm": version("mlx-lm")}))
"""
runtime_info = json.loads(subprocess.check_output([sys.executable, "-c", probe], text=True))
hardware = {
    "machine": subprocess.check_output(["sysctl", "-n", "hw.model"], text=True).strip(),
    "chip": subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip(),
    "macos": platform.mac_ver()[0], "architecture": platform.machine(),
    "python": sys.version, "python_executable": sys.executable,
    "unified_memory_bytes": psutil.virtual_memory().total,
    "available_memory_bytes_at_start": psutil.virtual_memory().available,
    "backend": BACKEND, "mlx_lm_revision_requested": MLX_LM_REVISION, **runtime_info,
}
(RUN_DIR / "hardware.json").write_text(json.dumps(hardware, indent=2), encoding="utf-8")
print(json.dumps(hardware, indent=2))
print("Unified memory (GiB):", round(hardware["unified_memory_bytes"] / 2**30, 1))
print("This environment check does not establish that the model will fit in memory.")


<a id="Data"></a>

## 3. Load the published dataset

`train.jsonl` contains **1,400 training examples**; `valid.jsonl` contains **100 validation
examples**. Each example already includes the payment rules and request in `prompt`, with a
direct JSON answer in `response`. No data generation, random split, or repair happens here.

The source is synthetic payment SFT data, not production payment records. Scenarios cover
promotions, rounding, channel recommendations, split payments, and error responses.
`bucket` labels describe the source scenarios; they are not independently verified coverage guarantees.

The validation split is used for before/after comparison. It is **not an independent test set**
and should not be used to claim final generalization after repeated hyperparameter tuning.


In [ ]:
from datasets import load_dataset
from recipe import check_splits

revision = DATA_REVISIONS[DATA_HUB]
if DATA_HUB == "modelscope":
    from modelscope.hub.snapshot_download import dataset_snapshot_download
    dataset_dir = Path(dataset_snapshot_download(
        DATASET_ID, revision=revision, allow_file_pattern=["train.jsonl", "valid.jsonl"],
    ))
    data_files = {"train": str(dataset_dir / "train.jsonl"), "validation": str(dataset_dir / "valid.jsonl")}
else:
    from huggingface_hub import hf_hub_download
    data_files = {split: hf_hub_download(DATASET_ID, filename=name, repo_type="dataset", revision=revision)
                  for split, name in (("train", "train.jsonl"), ("validation", "valid.jsonl"))}

dataset = load_dataset("json", data_files=data_files)
train_rows, valid_rows = list(dataset["train"]), list(dataset["validation"])
check_splits(train_rows, valid_rows)
print(dataset)
print("Dataset revision:", revision)


In [ ]:
from collections import Counter
from IPython.display import Markdown, display

print("Training buckets:", dict(Counter(row["bucket"] for row in train_rows)))
print("Validation buckets:", dict(Counter(row["bucket"] for row in valid_rows)))
example = train_rows[0]
print("Case:", example["case_id"], "| Bucket:", example["bucket"])
print("PROMPT:\n", example["prompt"])
print("TARGET:\n", json.dumps(example["ground_truth"], ensure_ascii=False, indent=2))


## 4. Prepare the model and check token lengths

This step downloads model metadata first, then the weights. A local checkpoint skips the download.
The stored weight size is reported below. Unified memory must also accommodate training
activations, caches, and the operating system; weight size alone is not a capacity guarantee.

AReno applies the model's chat template with thinking disabled. Only response tokens (including EOS)
contribute to the SFT loss. Every row is checked against the explicit budgets before model execution;
an oversized row stops the run instead of disappearing from the experiment.


In [ ]:
from modelscope import snapshot_download

if MODEL_PATH:
    model_path = Path(MODEL_PATH).expanduser().resolve()
    if not (model_path / "config.json").is_file():
        raise FileNotFoundError(f"Checkpoint config not found: {model_path}")
else:
    model_path = Path(snapshot_download(
        MODEL_ID, revision=MODEL_REVISION,
        allow_file_pattern=["*.json", "*.jinja", "*.py"],
    ))
model_config = json.loads((model_path / "config.json").read_text())
if model_config.get("model_type") != "bailing_hybrid" or model_config.get("quantization_config") or model_config.get("quantization"):
    raise ValueError("Use an unquantized Ling Tiny (bailing_hybrid) checkpoint for this recipe.")
index_path = model_path / "model.safetensors.index.json"
if index_path.is_file():
    weight_bytes = json.loads(index_path.read_text()).get("metadata", {}).get("total_size")
    if weight_bytes:
        print("Stored weights (GiB):", round(weight_bytes / 2**30, 2))
if not MODEL_PATH:
    model_path = Path(snapshot_download(MODEL_ID, revision=MODEL_REVISION))
print("Model:", model_path)


In [ ]:
from recipe import TARGET_MODULES, preflight

config = {
    "backend": BACKEND, "model_id": MODEL_ID, "model_path": str(model_path),
    "model_revision": MODEL_REVISION if not MODEL_PATH else "user-provided-local-checkpoint",
    "dataset_id": DATASET_ID, "data_hub": DATA_HUB, "data_revision": revision,
    "train_file": data_files["train"], "valid_file": data_files["validation"],
    "run_dir": str(RUN_DIR), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE, "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
    "seed": SEED, "max_prompt_tokens": MAX_PROMPT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS, "max_context_len": MAX_CONTEXT_LEN,
    "target_modules": list(TARGET_MODULES),
    "areno_commit": subprocess.check_output(["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True).strip(),
    "source_dirty": bool(subprocess.check_output(["git", "-C", str(ROOT), "status", "--porcelain"], text=True).strip()),
    "python": sys.version, "platform": platform.platform(),
}
token_report = preflight(config)
print(json.dumps(token_report, indent=2))
(RUN_DIR / "preflight.json").write_text(json.dumps(token_report, indent=2), encoding="utf-8")
config_path = RUN_DIR / "config.json"
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
(RUN_DIR / "environment.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8",
)
print("Attention LoRA targets:", TARGET_MODULES)


<a id="Baseline"></a>

## 5. Measure the base model

Each stage runs in a fresh Python process using `from areno import Trainer`. This releases
model memory between stages on the Mac. The base measurement attaches
a freshly initialized, zero-effect LoRA adapter; it performs no optimizer steps.

**Strict scoring:** the completion must be one JSON document, matching all fields, types,
values, and array order in the published answer. Object key order and whitespace do not matter.
Extra prose, Markdown fences, duplicate keys, missing/extra fields, or `1.0` in place of integer
cents fail exact match. JSON validity is reported separately and does not imply business correctness.


In [ ]:
import time


def run_stage(stage):
    command = [sys.executable, "-u", str(RECIPE / "recipe.py"), stage, str(config_path)]
    log_path = RUN_DIR / f"{stage}.log"
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            returncode = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode:
        raise RuntimeError(f"{stage} failed (exit {returncode}). See {log_path}")
    elapsed = time.perf_counter() - started
    timing_path = RUN_DIR / "stage-times.json"
    timings = json.loads(timing_path.read_text()) if timing_path.exists() else {}
    timings[stage] = {"wall_seconds": elapsed}
    timing_path.write_text(json.dumps(timings, indent=2), encoding="utf-8")
    print(f"{stage} completed in {elapsed / 60:.1f} minutes (includes worker setup and model loading).")

run_stage("before")
print((RUN_DIR / "before-summary.json").read_text())


<a id="Train"></a>

## 6. Train and save the LoRA adapter

AReno keeps the base weights frozen and updates the selected attention projections.
The worker uses the public `Trainer.train(batch, sft_loss_fn, mini_bs=1)` path, deterministically
shuffling only the training rows each epoch. Validation examples never enter the training batches.

<a id="AdapterSave"></a>

The final batch is followed by `Trainer.save_checkpoint(...)`. The saved adapter is a PEFT-style
artifact containing `adapter_config.json` and `adapter_model.safetensors`; it requires the same
base checkpoint. This is an adapter export, not an optimizer-state resume checkpoint.

Scalar metrics returned by the SDK are saved in `training.jsonl`; AReno also writes TensorBoard
events. No loss values are inferred by parsing console output.


In [ ]:
print("Optimizer steps:", EPOCHS * ((len(train_rows) + BATCH_SIZE - 1) // BATCH_SIZE))
run_stage("train")
from recipe import read_jsonl
training = read_jsonl(RUN_DIR / "training.jsonl")
print("Completed steps:", len(training))
print("First loss:", training[0]["loss"], "| Last loss:", training[-1]["loss"])
print("Adapter files:", [p.name for p in (RUN_DIR / "adapter").iterdir()])


<a id="Inference"></a>

## 7. Reload and compare

A new worker loads the base checkpoint plus the saved adapter through
`LoraConfig(adapter_path=...)`, then repeats greedy evaluation on the same 100 validation rows.
Successful inference here exercises the saved adapter rather than the training process's in-memory state.
This is a reload execution check, not a numerical round-trip equivalence test.

Before/after results measure the change within this run. CUDA and MLX need not produce identical
floating-point results, even with the same seed. Improvement is an experimental outcome, not assumed.


In [ ]:
run_stage("after")
before = json.loads((RUN_DIR / "before-summary.json").read_text())
after = json.loads((RUN_DIR / "after-summary.json").read_text())
table = ["| Validation metric | Before | After |", "| --- | ---: | ---: |"]
for key in ("rows", "valid_json", "exact_match", "at_token_limit"):
    table.append(f"| {key} | {before[key]} | {after[key]} |")
table.append(f"| Exact match rate | {before['exact_match_rate']:.1%} | {after['exact_match_rate']:.1%} |")
display(Markdown("\n".join(table)))
display(Markdown("### Results by source bucket"))
table = ["| Bucket | Rows | Before correct | After correct |", "| --- | ---: | ---: | ---: |"]
for bucket in sorted(before["buckets"]):
    old, new = before["buckets"][bucket], after["buckets"][bucket]
    table.append(f"| {bucket} | {old['rows']} | {old['exact_match']} | {new['exact_match']} |")
display(Markdown("\n".join(table)))


In [ ]:
# Inspect a reproducible sample and up to three remaining failures.
before_rows = {row["case_id"]: row for row in read_jsonl(RUN_DIR / "before.jsonl")}
after_rows = read_jsonl(RUN_DIR / "after.jsonl")
for row in [after_rows[0], *[r for r in after_rows[1:] if not r["exact_match"]][:3]]:
    print("\nCASE:", row["case_id"], "| BUCKET:", row["bucket"])
    print("BEFORE:", before_rows[row["case_id"]]["completion"])
    print("AFTER:", row["completion"])
    print("EXPECTED:", json.dumps(row["expected"], ensure_ascii=False))


<a id="Save"></a>

## 8. Keep your results

The run directory contains the configuration, environment versions, SDK metrics, raw completions,
summary scores, and adapter on your local Mac. `hardware.json` records the machine and runtime;
`preflight.json` records token lengths; `stage-times.json` records completed stages' wall time.
Stage time includes worker setup, model loading, and output writing, so it is not pure training throughput. Save this notebook in JupyterLab to
retain its cell outputs alongside your experiment artifacts. If a cell fails, inspect
its error output and the stage log in the run directory before retrying.

No upload happens automatically. Do not infer a production payment guarantee from synthetic
validation scores. For final evaluation after tuning, publish and use a separate held-out test split.

### Next experiments

Change one training parameter at a time and create a new run directory. Keep dataset revisions,
evaluation examples, and decoding settings fixed when comparing runs. An OOM is a capacity failure;
reduce batch concurrency or choose larger hardware before changing task token budgets.

### Troubleshooting

- **Wrong architecture or missing Metal:** use a native arm64 Python and a macOS version supported by MLX.
- **Missing Ling support:** rerun the pinned MLX-LM installation in the selected kernel environment.
- **Out of memory or heavy swap:** interrupt the running cell and close other memory-heavy apps.
  This recipe already uses one training row per microbatch and one evaluation request at a time;
  lowering `BATCH_SIZE` alone may not fix the peak. Keep token budgets unchanged for comparable runs.
- **Token budget failure:** inspect the reported case and lengths; explicitly increase the relevant
  budget and context length only when memory permits. No rows are silently truncated or removed.
- **Interrupted training:** only the final adapter is saved; start a new run rather than treating
  partial metrics as a completed epoch. Rerunning a stage overwrites its own log and outputs.

See the [Mac mini guide](MAC_MINI.md) for artifact reuse and reporting results.

### Reference

The accompanying Mac mini guide follows the setup-to-notebook structure of
[Unsloth's DGX Station guide](https://unsloth.ai/docs/blog/dgx-station).
The notebook organization is also inspired by the
[Unsloth Qwen3.5 Vision notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_5_%280_8B%29_Vision.ipynb#scrollTo=gib37dRGOWGF).
The training implementation here uses AReno. The dataset is licensed under Apache-2.0;
check the base model's model card for its separate terms.


In [ ]:
print("All run artifacts:", RUN_DIR)
print("Adapter:", RUN_DIR / "adapter")
print("Completed stage wall times:", (RUN_DIR / "stage-times.json").read_text())
print("Validation exact-match change (percentage points):",
      round(100 * (after["exact_match_rate"] - before["exact_match_rate"]), 2))
